# OpenRouter LLM Usage Analysis

The following notebook presents an analysis of global large language model (LLM) usage based on data collected from the [OpenRouter API platform](https://openrouter.ai). It demonstrates how API usage data can be processed to construct country-level indicators of model adoption and ecosystem trends.

The workflow illustrates how to transform raw usage logs into analytical indicators through a sequence of data processing steps including model classification, country–month aggregation, and index construction. The resulting dataset enables comparative analysis of global usage patterns for different types of language models.

In particular, the analysis focuses on three indicators:

- **Relative API token usage**, measuring the scale of LLM activity across countries
- **Open-source model usage**, capturing the share of tokens processed by open-weight models
- **Chinese model usage**, measuring the adoption of models developed by Chinese organisations

The notebook demonstrates how computational pipelines can be used to convert platform usage data into interpretable indicators and visualisations, including time-series charts and global maps.

## Data Sources

The analysis uses aggregated API usage data collected from the OpenRouter platform.

### OpenRouter Usage Data

- **Source:** OpenRouter API logs  
- **Description:** The dataset records monthly token consumption for models accessed through the OpenRouter platform. Each observation contains the country of the request, the model identifier, and the total number of tokens processed.

This dataset provides a unique view into the evolving landscape of large language model usage across countries and across different model ecosystems.

**Key variables include:**

- `month` — month of API usage  
- `country` — country where the request originated  
- `model_permaslug` — unique model identifier  
- `total_tokens` — number of tokens processed by the API

## Data Pre-processing

To ensure consistent and comparable indicators, several preprocessing steps are applied before analysis.

### Model Classification

Models are classified according to two analytical dimensions:

- **Open-source status** — whether the model provides open weights or open licensing  
- **Chinese origin** — whether the model is developed by organisations headquartered in China

Classification is performed using a combination of provider-level rules, exact model overrides, and heuristic checks based on model naming patterns.

Each model is assigned a `classification_source` label indicating whether the classification was derived from provider rules, manual overrides, heuristic inference, or fallback rules.

### Data Cleaning

Before aggregation, several cleaning steps are applied:

- Standardising month values to a consistent `YYYY-MM` format  
- Normalising country codes and handling missing values  
- Converting token counts to numeric values and removing invalid entries  
- Generating quality flags for missing or incomplete records

## Aggregation and Index Construction

After cleaning and classification, the dataset is aggregated to the **country × month** level.

For each country and month, the following metrics are computed:

- Total API token usage
- Token usage by open-source models
- Token usage by Chinese models

From these values, two usage shares are calculated:

- **Open-source share:** proportion of tokens processed by open-source models  
- **Chinese model share:** proportion of tokens processed by Chinese models

### Token Usage Index

Because API token usage grows rapidly over time, a logarithmic transformation is applied to stabilise the distribution:

$$
\text{token metric} = \log(1 + \text{total tokens})
$$

The transformed values are then scaled to a 0–100 index using min–max normalisation:

$$
\text{Token Index} =
\frac{x - x_{\min}}{x_{\max} - x_{\min}} \times 100
$$

### Share-Based Indices

The open-source and Chinese indices are derived directly from their respective shares:

- **Open-source index**

$$
\text{Open-source Index} = \text{open-source share} \times 100
$$

- **Chinese model index**

$$
\text{Chinese Index} = \text{Chinese share} \times 100
$$

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import pycountry

# ----------------------------
# Parameters
# ----------------------------
INPUT_FILE = "country_month_indices.csv"
OUTPUT_DIR = Path("piece4_outputs")
TOP_N_COUNTRIES_FOR_LINES = 15
EXCLUDE_UNKNOWN_COUNTRY = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Load and validate
# ----------------------------
df = pd.read_csv(INPUT_FILE)

required_cols = {
    "country",
    "month",
    "total_tokens",
    "token_index",
    "open_source_index",
    "chinese_index",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# ----------------------------
# Clean fields
# ----------------------------
df = df.copy()

df["country"] = (
    df["country"]
    .fillna("UNK")
    .astype(str)
    .str.strip()
    .str.upper()
    .replace("", "UNK")
)

df["month"] = df["month"].fillna("").astype(str).str.strip()

unique_months = pd.Series(df["month"].unique(), name="month")
month_lookup = pd.Series(
    pd.to_datetime(unique_months, errors="coerce").dt.to_period("M").values,
    index=unique_months
)
df["month_period"] = df["month"].map(month_lookup)

for col in ["total_tokens", "token_index", "open_source_index", "chinese_index"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["total_tokens"] = df["total_tokens"].fillna(0)
df.loc[df["total_tokens"] < 0, "total_tokens"] = 0

for col in ["token_index", "open_source_index", "chinese_index"]:
    df[col] = df[col].fillna(0).clip(lower=0, upper=100)

df = df.dropna(subset=["month_period"]).copy()

if EXCLUDE_UNKNOWN_COUNTRY:
    df = df.loc[df["country"] != "UNK"].copy()

if df.empty:
    raise ValueError("No rows available for visualisation after cleaning/filtering.")

df = df.sort_values(["month_period", "country"]).reset_index(drop=True)
df["month_label"] = df["month_period"].astype(str)

# ----------------------------
# ISO3 helper
# ----------------------------
def iso2_to_iso3(code: str):
    code = (code or "").strip().upper()

    manual = {
        "UK": "GBR",
        "EL": "GRC",
        "XK": "XKX",
    }
    if code in manual:
        return manual[code]

    if len(code) == 3 and code.isalpha():
        return code

    if len(code) != 2 or not code.isalpha():
        return None

    match = pycountry.countries.get(alpha_2=code)
    if match:
        return match.alpha_3

    fallback = {
        "US": "USA", "GB": "GBR", "DE": "DEU", "FR": "FRA", "IT": "ITA",
        "ES": "ESP", "PT": "PRT", "NL": "NLD", "BE": "BEL", "CH": "CHE",
        "AT": "AUT", "SE": "SWE", "NO": "NOR", "DK": "DNK", "FI": "FIN",
        "IE": "IRL", "PL": "POL", "CZ": "CZE", "HU": "HUN", "RO": "ROU",
        "BG": "BGR", "GR": "GRC", "TR": "TUR", "UA": "UKR", "RU": "RUS",
        "CA": "CAN", "MX": "MEX", "BR": "BRA", "AR": "ARG", "CL": "CHL",
        "CO": "COL", "PE": "PER", "AU": "AUS", "NZ": "NZL", "JP": "JPN",
        "CN": "CHN", "HK": "HKG", "TW": "TWN", "IN": "IND", "ID": "IDN",
        "MY": "MYS", "SG": "SGP", "TH": "THA", "VN": "VNM", "PH": "PHL",
        "KR": "KOR", "ZA": "ZAF", "NG": "NGA", "KE": "KEN", "GH": "GHA",
        "ET": "ETH", "TZ": "TZA", "UG": "UGA", "MW": "MWI", "ZM": "ZMB",
        "ZW": "ZWE", "MZ": "MOZ", "BW": "BWA", "NA": "NAM", "EG": "EGY",
        "MA": "MAR", "TN": "TUN", "DZ": "DZA", "AE": "ARE", "SA": "SAU",
        "IL": "ISR",
    }
    return fallback.get(code)

df["iso3"] = df["country"].map(iso2_to_iso3)

map_df = df.dropna(subset=["iso3"]).copy()
map_df = map_df.sort_values(["month_label", "country"]).reset_index(drop=True)

top_countries = (
    df.groupby("country", dropna=False)["total_tokens"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N_COUNTRIES_FOR_LINES)
    .index.tolist()
)

line_df = df.loc[df["country"].isin(top_countries)].copy()
line_df = line_df.sort_values(["month_period", "country"]).reset_index(drop=True)

if line_df.empty:
    raise ValueError("No countries available for line charts after selecting top countries.")

country_iso_lookup = (
    df[["country", "iso3"]]
    .drop_duplicates()
    .sort_values(["country", "iso3"], na_position="last")
)

## Trends in Model Usage Indicators
### Token Usage Index Trends

In [3]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import ColumnDataSource, HoverTool, Legend, Div
from bokeh.palettes import Category20
output_notebook()

# ── Parameters ────────────────────────────────────────────────────────────────
MIN_MONTHS = 6
MIN_TOKENS_FOR_PLOT = 1_000_000
TOP_N_COUNTRIES_FOR_LINES = 15

# ── Basic checks ──────────────────────────────────────────────────────────────
required_cols = {
    "country",
    "month",
    "total_tokens",
    "token_index",
    "open_source_index",
    "chinese_index",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in df: {sorted(missing)}")

plot_df = df.copy()

# month is already clean YYYY-MM string
plot_df["month"] = plot_df["month"].astype(str).str.strip()

# Standardise country
plot_df["country"] = (
    plot_df["country"]
    .fillna("UNK")
    .astype(str)
    .str.strip()
    .str.upper()
)

# Ensure numeric fields
for col in ["total_tokens", "token_index", "open_source_index", "chinese_index"]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")

plot_df = plot_df.dropna(
    subset=["total_tokens", "token_index", "open_source_index", "chinese_index"]
).copy()


# ── Filter 1: country must have >= MIN_MONTHS months of data ─────────────────
country_month_counts = plot_df.groupby("country")["month"].nunique()
active_countries = country_month_counts[country_month_counts >= MIN_MONTHS].index
df_active = plot_df[plot_df["country"].isin(active_countries)].copy()

# ── Filter 2: drop individual country-months with low token volume ───────────
df_active = df_active[df_active["total_tokens"] >= MIN_TOKENS_FOR_PLOT].copy()

# ── Filter 3: recheck month counts after token filter ────────────────────────
country_month_counts_post = df_active.groupby("country")["month"].nunique()
active_countries_post = country_month_counts_post[country_month_counts_post >= MIN_MONTHS].index
df_active = df_active[df_active["country"].isin(active_countries_post)].copy()

# ── Top 15 by total token volume — shared across all three charts ─────────────
top_by_tokens = (
    df_active.groupby("country")["total_tokens"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N_COUNTRIES_FOR_LINES)
    .index.tolist()
)

# ── Build chart dataframe — same countries for all three charts ───────────────
line_df = (
    df_active[df_active["country"].isin(top_by_tokens)]
    .sort_values(["month", "country"])
    .reset_index(drop=True)
)

# All months in sorted order — shared x-axis
all_months = sorted(line_df["month"].unique().tolist())

# ── Chart function ────────────────────────────────────────────────────────────
def make_bokeh_line_chart(data, y_col, title, y_axis_title, all_months):
    if data.empty:
        return Div(text=f"<b>{title}</b><br>No data available.")

    countries = sorted(data["country"].unique().tolist())

    palette = Category20[20] if len(countries) <= 20 else Category20[20] * (len(countries) // 20 + 1)
    color_map = {c: palette[i] for i, c in enumerate(countries)}

    p = figure(
        x_range=all_months,
        height=450,
        width=1300,
        title=title,
        toolbar_location="above",
        x_axis_label="Month",
        y_axis_label=y_axis_title,
    )

    p.xaxis.major_label_orientation = 0.8
    p.grid.grid_line_alpha = 0.3
    p.title.text_font_size = "13px"

    renderers = []
    for country in countries:
        sub = data[data["country"] == country].sort_values("month")

        source = ColumnDataSource(dict(
            month=sub["month"].tolist(),
            value=sub[y_col].tolist(),
            country=sub["country"].tolist(),
            total_tokens=sub["total_tokens"].tolist(),
            token_index=sub["token_index"].tolist(),
            open_source_index=sub["open_source_index"].tolist(),
            chinese_index=sub["chinese_index"].tolist(),
        ))

        line = p.line(
            x="month",
            y="value",
            source=source,
            line_width=2,
            color=color_map[country],
            name=country,
        )
        marker = p.scatter(
            x="month",
            y="value",
            source=source,
            size=6,
            color=color_map[country],
            name=country,
        )
        renderers.append((country, [line, marker]))

    legend = Legend(items=renderers, location="top_left")
    legend.click_policy = "hide"
    legend.label_text_font_size = "11px"
    p.add_layout(legend, "right")

    p.add_tools(HoverTool(
        tooltips=[
            ("Country", "@country"),
            ("Month", "@month"),
            ("Value", "@value{0.00}"),
            ("Total tokens", "@total_tokens{0,0}"),
            ("Token index", "@token_index{0.00}"),
            ("OS index", "@open_source_index{0.00}"),
            ("Chinese index", "@chinese_index{0.00}"),
        ],
        mode="mouse"
    ))

    return p

# ── Build charts ──────────────────────────────────────────────────────────────
p_token = make_bokeh_line_chart(
    line_df,
    y_col="token_index",
    title=f"Token volume index — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Token volume index (0–100)",
    all_months=all_months,
)

p_open = make_bokeh_line_chart(
    line_df,
    y_col="open_source_index",
    title=f"Open-source model share — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Open-source usage (%)",
    all_months=all_months,
)

p_chinese = make_bokeh_line_chart(
    line_df,
    y_col="chinese_index",
    title=f"Chinese model share — top {TOP_N_COUNTRIES_FOR_LINES} countries by total tokens",
    y_axis_title="Chinese model usage (%)",
    all_months=all_months,
)

# ── Show charts ───────────────────────────────────────────────────────────────
show(p_token)


Loading BokehJS ...

### Open-Source Model Usage Trends

In [4]:
show(p_open)

### Chinese Model Usage Trends

In [5]:
show(p_chinese)  

In [73]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

# ── Parameters ────────────────────────────────────────────────────────────────
MIN_TOKENS_FOR_MAP = 1_000_000

# ── Basic checks ──────────────────────────────────────────────────────────────
required_cols = {
    "country", "month", "total_tokens",
    "token_index", "open_source_index", "chinese_index",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in df: {sorted(missing)}")

# ── Prepare map dataframe ─────────────────────────────────────────────────────
map_df = df.copy()

map_df["country"] = (
    map_df["country"].fillna("").astype(str).str.strip().str.upper()
)
map_df["month"] = map_df["month"].fillna("").astype(str).str.strip()

for col in ["total_tokens", "token_index", "open_source_index", "chinese_index"]:
    map_df[col] = pd.to_numeric(map_df[col], errors="coerce")

map_df = map_df[map_df["total_tokens"] >= MIN_TOKENS_FOR_MAP].copy()
map_df = map_df.dropna(
    subset=["token_index", "open_source_index", "chinese_index"]
).copy()
map_df = map_df[map_df["country"].ne("") & map_df["country"].ne("UNK")].copy()

# ── ISO-2 to ISO-3 conversion ─────────────────────────────────────────────────
ISO2_TO_ISO3 = {
    "AF":"AFG","AL":"ALB","DZ":"DZA","AS":"ASM","AD":"AND","AO":"AGO","AI":"AIA","AQ":"ATA","AG":"ATG",
    "AR":"ARG","AM":"ARM","AW":"ABW","AU":"AUS","AT":"AUT","AZ":"AZE","BS":"BHS","BH":"BHR","BD":"BGD",
    "BB":"BRB","BY":"BLR","BE":"BEL","BZ":"BLZ","BJ":"BEN","BM":"BMU","BT":"BTN","BO":"BOL","BA":"BIH",
    "BW":"BWA","BR":"BRA","BN":"BRN","BG":"BGR","BF":"BFA","BI":"BDI","CV":"CPV","KH":"KHM","CM":"CMR",
    "CA":"CAN","KY":"CYM","CF":"CAF","TD":"TCD","CL":"CHL","CN":"CHN","CO":"COL","KM":"COM","CG":"COG",
    "CD":"COD","CR":"CRI","CI":"CIV","HR":"HRV","CU":"CUB","CW":"CUW","CY":"CYP","CZ":"CZE","DK":"DNK",
    "DJ":"DJI","DM":"DMA","DO":"DOM","EC":"ECU","EG":"EGY","SV":"SLV","GQ":"GNQ","ER":"ERI","EE":"EST",
    "SZ":"SWZ","ET":"ETH","FK":"FLK","FO":"FRO","FJ":"FJI","FI":"FIN","FR":"FRA","GF":"GUF","PF":"PYF",
    "GA":"GAB","GM":"GMB","GE":"GEO","DE":"DEU","GH":"GHA","GI":"GIB","GR":"GRC","GL":"GRL","GD":"GRD",
    "GP":"GLP","GU":"GUM","GT":"GTM","GG":"GGY","GN":"GIN","GW":"GNB","GY":"GUY","HT":"HTI","HN":"HND",
    "HK":"HKG","HU":"HUN","IS":"ISL","IN":"IND","ID":"IDN","IR":"IRN","IQ":"IRQ","IE":"IRL","IM":"IMN",
    "IL":"ISR","IT":"ITA","JM":"JAM","JP":"JPN","JE":"JEY","JO":"JOR","KZ":"KAZ","KE":"KEN","KI":"KIR",
    "KP":"PRK","KR":"KOR","KW":"KWT","KG":"KGZ","LA":"LAO","LV":"LVA","LB":"LBN","LS":"LSO","LR":"LBR",
    "LY":"LBY","LI":"LIE","LT":"LTU","LU":"LUX","MO":"MAC","MG":"MDG","MW":"MWI","MY":"MYS","MV":"MDV",
    "ML":"MLI","MT":"MLT","MH":"MHL","MQ":"MTQ","MR":"MRT","MU":"MUS","YT":"MYT","MX":"MEX","FM":"FSM",
    "MD":"MDA","MC":"MCO","MN":"MNG","ME":"MNE","MS":"MSR","MA":"MAR","MZ":"MOZ","MM":"MMR","NA":"NAM",
    "NR":"NRU","NP":"NPL","NL":"NLD","NC":"NCL","NZ":"NZL","NI":"NIC","NE":"NER","NG":"NGA","MK":"MKD",
    "MP":"MNP","NO":"NOR","OM":"OMN","PK":"PAK","PW":"PLW","PS":"PSE","PA":"PAN","PG":"PNG","PY":"PRY",
    "PE":"PER","PH":"PHL","PL":"POL","PT":"PRT","PR":"PRI","QA":"QAT","RE":"REU","RO":"ROU","RU":"RUS",
    "RW":"RWA","BL":"BLM","SH":"SHN","KN":"KNA","LC":"LCA","MF":"MAF","PM":"SPM","VC":"VCT","WS":"WSM",
    "SM":"SMR","ST":"STP","SA":"SAU","SN":"SEN","RS":"SRB","SC":"SYC","SL":"SLE","SG":"SGP","SX":"SXM",
    "SK":"SVK","SI":"SVN","SB":"SLB","SO":"SOM","ZA":"ZAF","SS":"SSD","ES":"ESP","LK":"LKA","SD":"SDN",
    "SR":"SUR","SE":"SWE","CH":"CHE","SY":"SYR","TW":"TWN","TJ":"TJK","TZ":"TZA","TH":"THA","TL":"TLS",
    "TG":"TGO","TO":"TON","TT":"TTO","TN":"TUN","TR":"TUR","TM":"TKM","TC":"TCA","TV":"TUV","UG":"UGA",
    "UA":"UKR","AE":"ARE","GB":"GBR","US":"USA","UY":"URY","UZ":"UZB","VU":"VUT","VE":"VEN","VN":"VNM",
    "VG":"VGB","VI":"VIR","YE":"YEM","ZM":"ZMB","ZW":"ZWE","XK":"XKX",
}

map_df["iso3"] = map_df["country"].map(ISO2_TO_ISO3)

missing_iso = sorted(map_df.loc[map_df["iso3"].isna(), "country"].unique().tolist())
if missing_iso:
    print("Warning: unmapped ISO-2 codes (dropped):", missing_iso)

map_df = map_df.dropna(subset=["iso3"]).copy()

# ── Chronological month order ─────────────────────────────────────────────────
map_df["month_period"] = pd.to_datetime(map_df["month"], errors="coerce").dt.to_period("M")
map_df = map_df.dropna(subset=["month_period"]).copy()
map_df["month"] = map_df["month_period"].astype(str)

map_df = (
    map_df.sort_values(["country", "month_period"])
    .drop_duplicates(subset=["country", "month"], keep="last")
    .copy()
)

month_order = (
    map_df[["month", "month_period"]]
    .drop_duplicates()
    .sort_values("month_period")["month"]
    .tolist()
)

map_df["month"] = pd.Categorical(map_df["month"], categories=month_order, ordered=True)
map_df = map_df.sort_values(["month", "iso3"]).copy()

print(f"Map rows: {len(map_df)}")
print(f"Months: {month_order}")
print(f"Countries: {map_df['iso3'].nunique()}")

# ── Style updater ─────────────────────────────────────────────────────────────
def style_fig(fig, title):
    fig.update_layout(
        autosize=True,
        height=820,
        margin=dict(l=20, r=120, t=70, b=130),
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(family="Arial, sans-serif", size=11, color="#333333"),
        title=dict(
            text=title,
            font=dict(size=15, family="Arial, sans-serif", color="#1A1A1A"),
            x=0.5,
            xanchor="center",
        ),
        geo=dict(
            domain=dict(x=[0.00, 0.88], y=[0.20, 1.00]),
            projection=dict(type="natural earth"),
            fitbounds=False,
            showframe=False,
            showcoastlines=True,
            coastlinewidth=0.5,
            coastlinecolor="#AAAAAA",
            showland=True,
            landcolor="rgb(240,240,240)",
            showocean=True,
            oceancolor="#DDEEFF",
            showlakes=False,
            bgcolor="white",
            showcountries=True,
            countrycolor="#DDDDDD",
            countrywidth=0.3,
        ),
        coloraxis_colorbar=dict(
            title=dict(side="top"),
            thickness=14,
            len=0.62,
            x=0.93,
            y=0.62,
            xanchor="left",
            yanchor="middle",
            tickfont=dict(size=10),
            outlinewidth=0,
        ),
    )

    fig.update_traces(
        marker_line_width=0.3,
        marker_line_color="white",
    )

    for frame in fig.frames:
        frame.layout.update(
            geo=dict(
                domain=dict(x=[0.02, 0.84], y=[0.20, 1.0]),
                projection=dict(type="natural earth"),
                fitbounds=False,
                showframe=False,
                showcoastlines=True,
                coastlinewidth=0.5,
                coastlinecolor="#AAAAAA",
                showland=True,
                landcolor="rgb(240,240,240)",
                showocean=True,
                oceancolor="#DDEEFF",
                showlakes=False,
                bgcolor="white",
                showcountries=True,
                countrycolor="#DDDDDD",
                countrywidth=0.3,
            )
        )

    if fig.layout.updatemenus:
        fig.layout.updatemenus[0].update(
            bgcolor="#F7F7F7",
            bordercolor="#CCCCCC",
            borderwidth=1,
            font=dict(size=11, color="#333333"),
            x=0.00,
            xanchor="left",
            y=0.05,
            yanchor="bottom",
        )

    if fig.layout.sliders:
        fig.layout.sliders[0].update(
            bgcolor="#F0F0F0",
            bordercolor="#CCCCCC",
            borderwidth=1,
            tickwidth=0,
            len=0.84,
            x=0.08,
            y=0.03,
            currentvalue=dict(
                prefix="Month: ",
                visible=True,
                xanchor="left",
                font=dict(size=11, color="#555555"),
            ),
            pad=dict(t=8, b=8),
        )

    return fig

# ── Map builder ───────────────────────────────────────────────────────────────
def make_choropleth(data, value_col, title, colorbar_title, hover_label, colorscale):
    fig = px.choropleth(
        data_frame=data,
        locations="iso3",
        color=value_col,
        hover_name="country",
        hover_data={
            "month": True,
            "total_tokens": ":,.0f",
            "token_index": ":.1f",
            "open_source_index": ":.1f",
            "chinese_index": ":.1f",
            "iso3": False,
        },
        animation_frame="month",
        color_continuous_scale=colorscale,
        range_color=(0, 100),
        projection="natural earth",
        labels={
            "month": "Month",
            "total_tokens": "Total tokens",
            "token_index": "Token index",
            "open_source_index": "Open-source %",
            "chinese_index": "Chinese model %",
        },
    )

    fig.update_traces(
        hovertemplate=(
            "<b>%{hovertext}</b><br>"
            "Month: %{customdata[0]}<br>"
            f"{hover_label}: %{{z:.1f}}<br>"
            "Total tokens: %{customdata[1]:,.0f}<br>"
            "Token index: %{customdata[2]:.1f}<br>"
            "Open-source %: %{customdata[3]:.1f}<br>"
            "Chinese model %: %{customdata[4]:.1f}<extra></extra>"
        ),
    )

    fig.update_layout(
        coloraxis_colorbar=dict(title=colorbar_title)
    )

    style_fig(fig, title)
    return fig

# ── Build maps ────────────────────────────────────────────────────────────────
fig_token = make_choropleth(
    data=map_df,
    value_col="token_index",
    title="Global map — relative API token usage over time",
    colorbar_title="Token index",
    hover_label="Token index",
    colorscale="Blues",
)

fig_open = make_choropleth(
    data=map_df,
    value_col="open_source_index",
    title="Global map — open-source model usage share over time",
    colorbar_title="Open-source usage (%)",
    hover_label="Open-source usage (%)",
    colorscale=[
        [0.0,  "#F7FCF5"],
        [0.25, "#BAE4B3"],
        [0.5,  "#74C476"],
        [0.75, "#31A354"],
        [1.0,  "#006D2C"],
    ],
)

fig_chinese = make_choropleth(
    data=map_df,
    value_col="chinese_index",
    title="Global map — Chinese model usage share over time",
    colorbar_title="Chinese model usage (%)",
    hover_label="Chinese model usage (%)",
    colorscale=[
        [0.0,  "#FFF5F0"],
        [0.25, "#FCBBA1"],
        [0.5,  "#FB6A4A"],
        [0.75, "#CB181D"],
        [1.0,  "#67000D"],
    ],
)

# ── Save responsive HTML ──────────────────────────────────────────────────────
out_dir = Path("_static/maps")
out_dir.mkdir(parents=True, exist_ok=True)

fig_token.write_html(
    out_dir / "token_map.html",
    include_plotlyjs="cdn",
    full_html=True,
    config={"responsive": True},
)

fig_open.write_html(
    out_dir / "open_source_map.html",
    include_plotlyjs="cdn",
    full_html=True,
    config={"responsive": True},
)

fig_chinese.write_html(
    out_dir / "chinese_map.html",
    include_plotlyjs="cdn",
    full_html=True,
    config={"responsive": True},
)

Map rows: 2601
Months: ['2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02']
Countries: 225


In [75]:
from pathlib import Path

out_dir = Path("_static/maps")
out_dir.mkdir(parents=True, exist_ok=True)

fig_token.write_html(
    out_dir / "token_map.html",
    include_plotlyjs=True,
    full_html=True,
)

fig_open.write_html(
    out_dir / "open_source_map.html",
    include_plotlyjs=True,
    full_html=True,
)

fig_chinese.write_html(
    out_dir / "chinese_map.html",
    include_plotlyjs=True,
    full_html=True,
)